In [1]:
import diopy
import pandas as pd
import scanpy as sc 
import scanpy.external as sce
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()

/bigdata/zlin/miniconda3/envs/scrna/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2023-05-17 13:58:30.451360: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-05-17 13:58:30.575380: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-05-17 13:58:31.211142: W tensorflow/compiler/xla/stream_executor/platf

scanpy==1.9.3 anndata==0.9.1 umap==0.5.3 numpy==1.21.6 scipy==1.9.1 pandas==1.5.3 scikit-learn==1.2.2 statsmodels==0.13.5 python-igraph==0.10.4 louvain==0.8.0 pynndescent==0.5.10


In [3]:
becker = diopy.input.read_h5(file='/bigdata/zlin/Melanoma_meta/data/Myeloids/becker.h5')
franken = diopy.input.read_h5(file='/bigdata/zlin/Melanoma_meta/data/Myeloids/franken.h5')
bassez1 = diopy.input.read_h5(file='/bigdata/zlin/Melanoma_meta/data/Myeloids/bassez1.h5')
bassez2 = diopy.input.read_h5(file='/bigdata/zlin/Melanoma_meta/data/Myeloids/bassez2.h5')
yost = diopy.input.read_h5(file='/bigdata/zlin/Melanoma_meta/data/Myeloids/yost.h5')
crc_li = diopy.input.read_h5(file='/bigdata/zlin/Melanoma_meta/data/Myeloids/crc_li.h5')
tnbc_li = diopy.input.read_h5(file='/bigdata/zlin/Melanoma_meta/data/Myeloids/tnbc_li.h5')

In [14]:
print(pd.crosstab(becker.obs['sample'],becker.obs['time_point']))
print(pd.crosstab(franken.obs['sample'],franken.obs['time_point']))
print(pd.crosstab(bassez1.obs['sample'],bassez1.obs['time_point']))
print(pd.crosstab(bassez2.obs['sample'],bassez2.obs['time_point']))
print(pd.crosstab(yost.obs['sample'],yost.obs['time_point']))
print(pd.crosstab(crc_li.obs['sample'],crc_li.obs['time_point']))
print(pd.crosstab(tnbc_li.obs['sample'],tnbc_li.obs['time_point']))

time_point  Before   On
sample                 
P1_Before      305    0
P1_On            0  179
P10_Before      34    0
P11_Before     401    0
P11_On           0   66
P12_Before     151    0
P12_On           0  296
P2_Before       18    0
P2_On            0   74
P3_Before      254    0
P3_On            0  356
P4_Before       31    0
P5_Before       12    0
P5_On            0  188
P6_Before      141    0
P6_On            0    7
P7_Before       94    0
P7_On            0  146
P8_Before        8    0
P9_Before       94    0
P9_On            0  166
time_point     On   Pre
sample                 
1_On         1946     0
1_Pre           0  1086
10_On         809     0
10_Pre          0   398
11_On          64     0
11_Pre          0   922
12_On        2475     0
12_Pre          0  3006
13_On        1172     0
13_Pre          0  1170
14_On         572     0
14_Pre          0  1798
15_On         937     0
15_Pre          0  1758
16_On         237     0
17_On         814     0
17_Pre          

In [15]:
datasets = [becker, yost, bassez1, bassez2, franken, crc_li, tnbc_li]

In [16]:
def rm_patients(adata):
    ct = pd.crosstab(adata.obs['patient'], adata.obs['time_point'])
    rm = ct[(ct.iloc[:, 0] < 50) | (ct.iloc[:, 1] < 50)]
    return rm.index.tolist()

In [20]:
# remove samples with insufficient cell numbers
rm_becker = rm_patients(becker)
rm_franken = rm_patients(franken)
rm_bassez1 = rm_patients(bassez1)
rm_bassez2 = rm_patients(bassez2)
rm_yost = rm_patients(yost)
rm_crc_li = rm_patients(crc_li)
rm_tnbc_li = rm_patients(tnbc_li)
print(rm_becker, rm_franken, rm_bassez1, rm_bassez2, rm_yost, rm_crc_li, rm_tnbc_li)

['P10', 'P2', 'P4', 'P5', 'P6', 'P8'] [16, 20] ['BIOKEY_14', 'BIOKEY_20', 'BIOKEY_22', 'BIOKEY_23', 'BIOKEY_25', 'BIOKEY_29'] ['BIOKEY_32', 'BIOKEY_36', 'BIOKEY_38', 'BIOKEY_39', 'BIOKEY_42'] ['su003', 'su004', 'su007', 'su009', 'su012'] ['P28', 'P30', 'P31'] []


In [21]:
becker = becker[~becker.obs['patient'].isin(rm_becker)]
franken = franken[~franken.obs['patient'].isin(rm_franken)]
bassez1 = bassez1[~bassez1.obs['patient'].isin(rm_bassez1)]
bassez2 = bassez2[~bassez2.obs['patient'].isin(rm_bassez2)]
yost = yost[~yost.obs['patient'].isin(rm_yost)]
crc_li = crc_li[~crc_li.obs['patient'].isin(rm_crc_li)]
tnbc_li = tnbc_li[~tnbc_li.obs['patient'].isin(rm_tnbc_li)]

In [22]:
dataset_list = [becker, yost,bassez1, bassez2, franken, crc_li, tnbc_li]

In [23]:
dataset_list[0] = dataset_list[0][~dataset_list[0].obs['celltype_sc_r2'].isin(['Mast_KIT'])]
for i in range(len(dataset_list)):
    dataset_list[i].obs['celltype_subtype'] = dataset_list[i].obs['celltype_sc_r2'].astype(str)
    dataset_list[i].obs.loc[dataset_list[i].obs['celltype_subtype'].str.contains('cDC2'),'celltype_subtype'] = 'cDC2'

/tmp/ipykernel_3194558/2793487121.py:3: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  dataset_list[i].obs['celltype_subtype'] = dataset_list[i].obs['celltype_sc_r2'].astype(str)
/tmp/ipykernel_3194558/2793487121.py:3: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  dataset_list[i].obs['celltype_subtype'] = dataset_list[i].obs['celltype_sc_r2'].astype(str)


In [42]:
pd.crosstab(becker.obs['celltype_sc_r2'],becker.obs['sample'])

sample,P1_Before,P1_On,P11_Before,P11_On,P12_Before,P12_On,P3_Before,P3_On,P7_Before,P7_On,P9_Before,P9_On
celltype_sc_r2,,,,,,,,,,,,
cDC1_CLEC9A,0,3,3,1,2,1,27,21,2,1,1,1
cDC2_CD1A,6,0,13,3,7,12,30,11,5,5,3,3
cDC2_CXCL9,13,7,15,0,2,15,10,33,3,3,3,11
cDC2_CXCR4hi,2,0,2,0,0,2,3,1,1,0,0,0
cDC2_FCN1,8,1,2,0,0,5,3,0,1,1,0,2
cDC2_IL1B,0,0,0,0,0,0,3,0,0,0,0,0
cDC2_ISG15,0,0,2,0,0,0,0,1,0,0,0,1
cDC3_LAMP3,6,6,5,0,0,4,42,22,1,0,0,0
Macro_C1QC,38,8,82,21,62,49,23,56,19,46,36,23


In [52]:
becker.obs['celltype_count'] = becker.obs.groupby(['sample','celltype_sc_r2'])['celltype_sc_r2'].transform('count')
becker.obs['sample_total'] = becker.obs.groupby(['sample'])['celltype_sc_r2'].transform('count')
becker.obs['celltype_proportion'] = becker.obs['celltype_count'] / becker.obs['sample_total']

In [ ]:
becker.obs['celltype_count'] = becker.obs.groupby(['sample','celltype_sc_r2'])['celltype_sc_r2'].transform('count')
becker.obs['sample_total'] = becker.obs.groupby(['sample'])['celltype_sc_r2'].transform('count')
becker.obs['celltype_proportion'] = becker.obs['celltype_count'] / becker.obs['sample_total']

In [82]:
# Step 2: pivot dataframe to have celltypes as columns with their proportions as values
pivot_df = becker.obs.pivot_table(values='celltype_proportion', index=['patient', 'time_point','response','treatment'], columns='celltype_sc_r2')
pivot_df.fillna(0.001, inplace=True)
pivot_df

# pivot_df.reset_index(inplace=True)

# pre_df = pivot_df[pivot_df['time_point'] == 'Before'].set_index('patient')

# on_df = pivot_df[pivot_df['time_point'] == 'On'].set_index('patient')
# proportion_columns = [col for col in pivot_df.columns if col not in ['patient', 'time_point','response','treatment']]
# fold_change_df = on_df[proportion_columns].divide(pre_df[proportion_columns]).reset_index()
# fold_change_df

celltype_sc_r2                         cDC1_CLEC9A  cDC2_CD1A  cDC2_CXCL9  \
patient time_point response treatment                                       
P1      Before     PR       PD1+CTLA4     0.001000   0.019672    0.042623   
        On         PR       PD1+CTLA4     0.016760   0.001000    0.039106   
P11     Before     SD       PD1           0.007481   0.032419    0.037406   
        On         SD       PD1           0.015152   0.045455    0.001000   
P12     Before     PD       PD1           0.013245   0.046358    0.013245   
        On         PD       PD1           0.003378   0.040541    0.050676   
P3      Before     SD       PD1+CTLA4     0.106299   0.118110    0.039370   
        On         SD       PD1+CTLA4     0.058989   0.030899    0.092697   
P7      Before     NE       PD1           0.021277   0.053191    0.031915   
        On         NE       PD1           0.006849   0.034247    0.020548   
P9      Before     PD       PD1           0.010638   0.031915    0.031915   
        On         PD       PD1           0.006024   0.018072    0.066265   

celltype_sc_r2                         cDC2_CXCR4hi  cDC2_FCN1  cDC2_IL1B  \
patient time_point response treatment                                       
P1      Before     PR       PD1+CTLA4      0.006557   0.026230   0.001000   
        On         PR       PD1+CTLA4      0.001000   0.005587   0.001000   
P11     Before     SD       PD1            0.004988   0.004988   0.001000   
        On         SD       PD1            0.001000   0.001000   0.001000   
P12     Before     PD       PD1            0.001000   0.001000   0.001000   
        On         PD       PD1            0.006757   0.016892   0.001000   
P3      Before     SD       PD1+CTLA4      0.011811   0.011811   0.011811   
        On         SD       PD1+CTLA4      0.002809   0.001000   0.001000   
P7      Before     NE       PD1            0.010638   0.010638   0.001000   
        On         NE       PD1            0.001000   0.006849   0.001000   
P9      Before     PD       PD1            0.001000   0.001000   0.001000   
        On         PD       PD1            0.001000   0.012048   0.001000   

celltype_sc_r2                         cDC2_ISG15  cDC3_LAMP3  Macro_C1QC  \
patient time_point response treatment                                       
P1      Before     PR       PD1+CTLA4    0.001000    0.019672    0.124590   
        On         PR       PD1+CTLA4    0.001000    0.033520    0.044693   
P11     Before     SD       PD1          0.004988    0.012469    0.204489   
        On         SD       PD1          0.001000    0.001000    0.318182   
P12     Before     PD       PD1          0.001000    0.001000    0.410596   
        On         PD       PD1          0.001000    0.013514    0.165541   
P3      Before     SD       PD1+CTLA4    0.001000    0.165354    0.090551   
        On         SD       PD1+CTLA4    0.002809    0.061798    0.157303   
P7      Before     NE       PD1          0.001000    0.010638    0.202128   
        On         NE       PD1          0.001000    0.001000    0.315068   
P9      Before     PD       PD1          0.001000    0.001000    0.382979   
        On         PD       PD1          0.006024    0.001000    0.138554   

celltype_sc_r2                         Macro_FN1  ...  Macro_INHBA  \
patient time_point response treatment             ...                
P1      Before     PR       PD1+CTLA4   0.001000  ...     0.003279   
        On         PR       PD1+CTLA4   0.001000  ...     0.005587   
P11     Before     SD       PD1         0.007481  ...     0.002494   
        On         SD       PD1         0.001000  ...     0.001000   
P12     Before     PD       PD1         0.006623  ...     0.001000   
        On         PD       PD1         0.003378  ...     0.001000   
P3      Before     SD       PD1+CTLA4   0.001000  ...     0.001000   
        On         SD       PD1+CTLA4   0.001000  ...     0.001000   
P7      Before     NE       PD1         0.001000  ...     0.010